In [1]:
from JointTemporalModel import JointTemporalModel
import torch, math
from torch.optim import AdamW
from transformers import get_cosine_schedule_with_warmup
from Utils import collator, TemporalDataset
from torch.utils.data import DataLoader

d:\GeoTKG\venv\Lib\site-packages\huggingface_hub\file_download.py:945: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [2]:
label2id_ner = {"B-DATE":0, "B-DURATION":1, "B-EVENT": 2, "B-TIME": 3, "I-DATE":4, "I-DURATION":5, "I-EVENT":6, "I-TIME":7, "O":8}
id2label_ner = {0: 'B-DATE', 1: 'B-DURATION', 2: 'B-EVENT', 3: 'B-TIME', 4: 'I-DATE', 5: 'I-DURATION', 6: 'I-EVENT', 7: 'I-TIME', 8: 'O'}
label2id_ee = {"AFTER": 0, "BEFORE": 1, "CONTAINS": 2, "DURING":3, "EQUALS":4, "IDENTITY":5, "OVERLAPS":6}
id2label_ee = {0: 'AFTER', 1: 'BEFORE', 2: 'CONTAINS', 3: 'DURING', 4: 'EQUALS', 5: 'IDENTITY', 6: 'OVERLAPS'}
cleandata_path = "D:\\GeoTKG\\cleandata\\tie\\"
def collate_fn(examples):
    return collator(examples, label2id_ner=label2id_ner, label2id_ee=label2id_ee)
train = TemporalDataset(cleandata_path + "train.json")
eval = TemporalDataset(cleandata_path + "eval.json")
train_loader = DataLoader(train, batch_size=8, shuffle=True, collate_fn=collate_fn)
eval_loader = DataLoader(eval, batch_size=8, shuffle=False, collate_fn=collate_fn)

In [3]:
NUM_EPOCHS = 50
ENC_LR = 5e-5
NONENC_LR = 1e-3
WARMUP_EPOCHS = 5
LOGGING_STEPS = 200
EVALUATION_EPOCHS = 10
BASE_ENC_MODEL = "roberta-base"
GRAD_ACCUM = 4
HEADS = 4
WEIGHT_DECAY = 0.01

In [4]:
device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')

model = JointTemporalModel(base=BASE_ENC_MODEL, num_ner=len(label2id_ner), ee_labels=len(label2id_ee), heads=HEADS).to(device)

for p in model.enc.parameters(): p.requires_grad = False

optimizer = AdamW([
        {"params": [p for n,p in model.named_parameters() if n.startswith("enc.")], "lr": ENC_LR},
        {"params": [p for n,p in model.named_parameters() if not n.startswith("enc.")], "lr": NONENC_LR},
    ], weight_decay=0.01)

steps_per_epoch = math.ceil(len(train_loader) / GRAD_ACCUM)
num_train_steps = steps_per_epoch * NUM_EPOCHS
num_warmup_steps = steps_per_epoch * WARMUP_EPOCHS

sched = get_cosine_schedule_with_warmup(optimizer, num_warmup_steps, num_train_steps)

scaler = torch.amp.GradScaler(enabled=(device.type=='cuda'))

def make_optim(unfrozen: bool):
    groups = []
    if unfrozen:
        groups.append({"params": model.enc.parameters(), "lr": ENC_LR})
    else:
        # keep enc group empty or skip entirely; either is fine
        pass
    nonenc = [p for n, p in model.named_parameters() if not n.startswith("enc.")]
    groups.append({"params": nonenc, "lr": NONENC_LR})
    return AdamW(groups, weight_decay=WEIGHT_DECAY)

Some weights of the model checkpoint at roberta-base were not used when initializing RobertaModel: ['lm_head.layer_norm.bias', 'lm_head.layer_norm.weight', 'lm_head.dense.weight', 'lm_head.bias', 'lm_head.dense.bias']
- This IS expected if you are initializing RobertaModel from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing RobertaModel from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-base and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
global_step = 0
history = {"loss": [], "ner_loss":[], "ca_loss":[], "ee_loss":[], "ner_f1": [], "ptr_acc": [], "ee_f1": [], "lr": []}
for epoch in range(NUM_EPOCHS):
    model.train()
    optimizer.zero_grad(set_to_none=True)

    for step, batch in enumerate(train_loader):
        batch = {k: (v.to(device) if torch.is_tensor(v) else v) for k, v in batch.items()}
        print(global_step)

        ctx = (torch.autocast(device_type='cuda', dtype=torch.float16))
        with ctx:
            # Ensure your model.forward signature matches these keys
            out = model(
                input_ids=batch["input_ids"],
                attention_mask=batch["attention_mask"],
                ev_starts=batch["ev_starts"], ev_ends=batch["ev_ends"], ev_mask=batch["ev_mask"],
                ti_starts=batch["ti_starts"], ti_ends=batch["ti_ends"], ti_mask=batch["ti_mask"],
                ner_gold_labels=batch["ner_labels"],
                ev_ti_gold=batch["ev_ti_gold"],
                ee_rel_gold=batch["ee_triples"],
                ee_mask=batch["ee_mask"],
            )
            loss = out["loss"] / GRAD_ACCUM

        scaler.scale(loss).backward()

        if (step + 1) % GRAD_ACCUM == 0:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)
            sched.step()
            global_step += 1
            
        if global_step % LOGGING_STEPS == 0:
            print(f"LOSS: {loss.item():.4f}, STEP: {global_step}")
            history["loss"].append(loss.item())
            history["ner_loss"].append(out["ner_loss"].item())
            history["ca_loss"].append(out["ca_loss"].item())
            history["ee_loss"].append(out["ee_loss"].item())
            history['lr'].append(optimizer.param_groups[0]['lr'])
    # ---- unfreeze after warmup epochs ----
    if epoch + 1 == WARMUP_EPOCHS:
        for p in model.enc.parameters():
            p.requires_grad = True
        # rebuild optimizer & scheduler for the remaining steps
        optimizer = make_optim(unfrozen=True)
        remaining_steps = steps_per_epoch * (NUM_EPOCHS - (epoch + 1))
        warmup_rem = 0  # already warmed up; or set a small extra warmup if you like
        sched = get_cosine_schedule_with_warmup(optimizer, warmup_rem, remaining_steps)

    # ---- validation ----
    if (epoch + 1) % EVALUATION_EPOCHS == 0:
        model.eval()
        with torch.no_grad():
            batch_evaluation = model.evaluate_dataloader(eval_loader, id2label_ner, id2label_ee)
            history["ner_f1"].append(batch_evaluation["ner_f1"])
            history["ptr_acc"].append(batch_evaluation["ptr_acc"])
            history["ee_f1"].append(batch_evaluation["ee_f1"])
        print(f"ep{epoch+1}: NER F1={batch_evaluation['ner_f1']:.4f}  PTR@1={batch_evaluation['ptr_acc']:.4f}  EE mF1={batch_evaluation['ee_f1']:.4f}")

0
LOSS: 6.4312, STEP: 0
0
LOSS: 4.2908, STEP: 0
0
LOSS: 5.2074, STEP: 0
0
1
1
1
1
2
2
2
2
3
3
3
3
4
4
4
4
5
5
5
5
6
6
6
6
7
7
7
7
8
8
8
8
9
9
9
9
10
10
10
10
11
11
11
11
12
12
12
12
13
13
13
13
14
14
14
14
15
15
15
15
16
16
16
16
17
17
17
17
18
18
18
18
19
19
19
19
20
20
20
20
21
21
21
21
22
22
22
22
23
23
23
23
24
24
24
24
25
25
25
25
26
26
26
26
27
27
27
27
28
28
28
28
29
29
29
29
30
30
30
30
31
31
31
31
32
32
32
32
33
33
33
33
34
34
34
34
35
35
35
35
36
36
36
36
37
37
37
37
38
38
38
38
39
39
39
39
40
40
40
40
41
41
41
41
42
42
42
42
43
43
43
43
44
44
44
44
45
45
45
45
46
46
46
46
47
47
47
47
48
48
48
48
49
49
49
49
50
50
50
50
51
51
51
51
52
52
52
52
53
53
53
53
54
54
54
54
55
55
55
55
56
56
56
56
57
57
57
57
58
58
58
58
59
59
59
59
60
60
60
60
61
61
61
61
62
62
62
62
63
63
63
63
64
64
64
64
65
65
65
65
66
66
66
66
67
67
67
67
68
68
68
68
69
69
69
69
70
70
70
70
71
71
71
71
72
72
72
72
73
73
73
73
74
74
74
74
75
75
75
75
76
76
76
76
77
77
77
77
78
78
78
78
79
79
79
79
80
80
80
80
81

KeyboardInterrupt: 

In [ ]:
import matplotlib.pyplot as plt

plt.plot(history["ner_f1"], label="NER F1")
plt.plot(history["ptr_acc"], label="Pointer Accuracy")
plt.plot(history["ee_f1"], label="EE F1")
plt.xlabel("Steps")
plt.ylabel("Score")
plt.legend()
plt.show()

In [ ]:
plt.plot(history["lr"], label="LR")
plt.xlabel("Steps")
plt.ylabel("LR")
plt.legend()
plt.show()

In [ ]:
plt.plot(history["loss"], label="Loss")
plt.show()